# 陈小群战法 - 市场环境判断（情绪周期）

> **策略来源**: 陈小群游资战法  
> **可靠性评级**: B级（中高可靠性）  
> **更新时间**: 2026-01-13

---

## 📊 功能说明

本Notebook用于判断市场情绪周期，是陈小群战法的基础步骤。

### 判断标准

| 周期阶段 | 涨停家数 | 连板高度 | 炸板率 | 资金流向 | 仓位策略 |
|---------|---------|---------|--------|---------|---------|
| **退潮期** | <10只 | <3板 | >40% | 净流出 | **0%** 空仓等待 |
| **启动期** | 10-30只 | 3-4板 | 10-20% | 小幅净流入 | **10%** 首板卡位术 |
| **加速期** | 30-60只 | 4-6板 | 15-25% | 大幅净流入 | **50%+** 龙头战法 |
| **过热期** | >60只 | >7板 | >30% | 极度净流入 | **30-50%** 逐步减仓 |

---

In [16]:
# 设置输出默认可滚动（限制最大高度）
from IPython.display import HTML, display

# 设置 Jupyter Notebook 输出区域的最大高度，超出部分可滚动
display(HTML("""
<style>
    /* 设置输出区域默认最大高度为600px，超出可滚动 */
    .jp-OutputArea-output {
        max-height: 600px !important;
        overflow-y: auto !important;
    }
    
    /* JupyterLab 特定样式 */
    .jp-Cell-outputArea {
        max-height: 600px !important;
        overflow-y: auto !important;
    }
    
    /* 单个输出项的最大高度 */
    .jp-OutputArea-child {
        max-height: 600px !important;
        overflow-y: auto !important;
    }
    
    /* 确保滚动条样式美观 */
    .jp-OutputArea-output::-webkit-scrollbar,
    .jp-Cell-outputArea::-webkit-scrollbar {
        width: 8px;
    }
    
    .jp-OutputArea-output::-webkit-scrollbar-track,
    .jp-Cell-outputArea::-webkit-scrollbar-track {
        background: #f1f1f1;
        border-radius: 4px;
    }
    
    .jp-OutputArea-output::-webkit-scrollbar-thumb,
    .jp-Cell-outputArea::-webkit-scrollbar-thumb {
        background: #888;
        border-radius: 4px;
    }
    
    .jp-OutputArea-output::-webkit-scrollbar-thumb:hover,
    .jp-Cell-outputArea::-webkit-scrollbar-thumb:hover {
        background: #555;
    }
</style>
"""))

print("✅ 输出区域已设置为可滚动（最大高度600px）")

✅ 输出区域已设置为可滚动（最大高度600px）


## 🔧 环境初始化

In [17]:
import sys
from pathlib import Path

# 自动检测项目根目录
current_dir = Path.cwd()
project_root = None
for parent in [current_dir] + list(current_dir.parents):
    if (parent / 'core').exists() and (parent / 'config').exists():
        project_root = parent
        break

if project_root is None:
    project_root = Path('/home/taotao/.cursor/worktrees/TRQuant/ope')

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 使用统一环境初始化
from notebooks.lib import setup_research_environment
env = setup_research_environment(verbose=True)

2026-01-14 07:45:03,547 - notebooks.lib.research_init - INFO - ✅ 项目根目录: /home/taotao/.cursor/worktrees/TRQuant/ope
2026-01-14 07:45:03,552 - notebooks.lib.research_init - INFO - ✅ 加载配置: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/research.yaml


研究环境状态
项目根目录: /home/taotao/.cursor/worktrees/TRQuant/ope
Python 版本: 3.12.3
当前时间: 2026-01-14 07:45:03
JQData 客户端: ⏳ 未初始化
趋势分析器: ⏳ 未初始化
评估引擎: ⏳ 未初始化


## 📊 数据源检测

In [18]:
from notebooks.lib import ErrorBoundary

# 检测JQData连接
jqdata_status = "❌ 未连接"
jq = None
with ErrorBoundary("检测JQData连接", suppress=True) as eb:
    jq = env.get_jqdata_client()
    if jq and hasattr(jq, 'is_authenticated') and jq.is_authenticated():
        jqdata_status = "✅ 已连接"
    elif jq:
        jqdata_status = "✅ 已连接"

print(f"JQData状态: {jqdata_status}")

# 检测AKShare
akshare_status = "❌ 未安装"
try:
    import akshare as ak
    akshare_status = "✅ 已安装"
except ImportError:
    pass

print(f"AKShare状态: {akshare_status}")

2026-01-14 07:45:03,556 - config.config_manager - INFO - 加载配置成功: jqdata_config.json
2026-01-14 07:45:03,557 - jqdata.auth - INFO - 聚宽认证成功: 13327806797
2026-01-14 07:45:03,557 - jqdata.client - INFO - 正在检测账号数据权限...
2026-01-14 07:45:05,683 - jqdata.client - INFO - ✅ 通过 get_account_info() 检测到账号权限: 数据模式: 实时, 范围: 2005-01-01 至 2026-01-14
2026-01-14 07:45:05,683 - notebooks.lib.research_init - INFO - ✅ JQData 客户端初始化成功


JQData状态: ✅ 已连接
AKShare状态: ✅ 已安装


## 📈 1. 获取涨停板数据

In [19]:
import akshare as ak
import pandas as pd
from datetime import datetime, timedelta
import numpy as np

# 获取当前日期
today = datetime.now()
today_str = today.strftime('%Y%m%d')
today_display = today.strftime('%Y-%m-%d')

print(f"📅 当前日期: {today_display}")
print()

# 工具函数：识别股票交易所类型并转换为JQData格式
def identify_exchange_and_convert(code):
    """
    识别股票交易所类型并转换为JQData格式
    
    返回:
        (jq_code, exchange_type, is_valid)
        - jq_code: JQData格式代码（如果支持），否则为None
        - exchange_type: 交易所类型（'XSHE'/'XSHG'/'BSE'/'OTHER'）
        - is_valid: 是否可以在JQData中查询
    """
    if not code:
        return None, 'OTHER', False
    
    code_str = str(code)
    if len(code_str) != 6:
        return None, 'OTHER', False
    
    # 北交所股票（92开头）
    if code_str.startswith('92'):
        return None, 'BSE', False  # 北交所，JQData不支持
    
    # 深市股票（00、30开头）
    if code_str.startswith('00') or code_str.startswith('30'):
        return f"{code_str}.XSHE", 'XSHE', True
    
    # 沪市股票（60、68开头）
    elif code_str.startswith('60') or code_str.startswith('68'):
        return f"{code_str}.XSHG", 'XSHG', True
    
    # 其他格式
    return None, 'OTHER', False

# 兼容旧函数名
def convert_code_to_jq(code):
    """将AKShare股票代码转换为JQData格式（兼容旧代码）"""
    jq_code, _, is_valid = identify_exchange_and_convert(code)
    return jq_code if is_valid else None

# 获取涨停板数据
print("📊 正在获取涨停板数据...")
try:
    limit_up_data = ak.stock_zt_pool_em(date=today_str)
    if limit_up_data is not None and not limit_up_data.empty:
        limit_up_count = len(limit_up_data)
        
        # 识别交易所类型并转换代码
        exchange_info = limit_up_data['代码'].apply(identify_exchange_and_convert)
        limit_up_data['jq_code'] = exchange_info.apply(lambda x: x[0])
        limit_up_data['exchange_type'] = exchange_info.apply(lambda x: x[1])
        limit_up_data['jqdata_supported'] = exchange_info.apply(lambda x: x[2])
        
        # 分类统计
        jqdata_stocks = limit_up_data[limit_up_data['jqdata_supported'] == True].copy()
        bse_stocks = limit_up_data[limit_up_data['exchange_type'] == 'BSE'].copy()
        other_stocks = limit_up_data[(limit_up_data['exchange_type'] != 'BSE') & (limit_up_data['jqdata_supported'] == False)].copy()
        
        print(f"✅ 获取成功！涨停家数: {limit_up_count}只")
        print(f"✅ JQData支持: {len(jqdata_stocks)}只（沪深A股）")
        if len(bse_stocks) > 0:
            print(f"📊 北交所股票: {len(bse_stocks)}只（将使用AKShare数据补充）")
        if len(other_stocks) > 0:
            print(f"⚠️  其他格式: {len(other_stocks)}只（无法处理）")
        
        # 显示北交所股票详情
        if len(bse_stocks) > 0:
            print(f"\n" + "=" * 80)
            print(f"📋 北交所股票列表（共 {len(bse_stocks)} 只）")
            print("=" * 80)
            
            bse_display_cols = ['代码', '名称']
            if '最新价' in bse_stocks.columns:
                bse_display_cols.append('最新价')
            if '涨跌幅' in bse_stocks.columns:
                bse_display_cols.append('涨跌幅')
            if '换手率' in bse_stocks.columns:
                bse_display_cols.append('换手率')
            if '连板数' in bse_stocks.columns:
                bse_display_cols.append('连板数')
            
            available_cols = [c for c in bse_display_cols if c in bse_stocks.columns]
            print(bse_stocks[available_cols].to_string(index=False))
            
            print(f"\n💡 北交所股票说明:")
            print(f"   • 代码格式：92开头（北京证券交易所）")
            print(f"   • JQData支持：❌ 不支持（JQData目前不支持北交所股票）")
            print(f"   • 数据补充：✅ 将使用AKShare历史数据计算连板高度")
            print(f"   • 影响分析：北交所股票通常市值较小，对整体市场情绪影响有限")
        
        # 显示其他无法处理的股票
        if len(other_stocks) > 0:
            print(f"\n" + "=" * 80)
            print(f"⚠️  其他无法处理的股票（共 {len(other_stocks)} 只）")
            print("=" * 80)
            
            other_display_cols = ['代码', '名称']
            if '最新价' in other_stocks.columns:
                other_display_cols.append('最新价')
            if '涨跌幅' in other_stocks.columns:
                other_display_cols.append('涨跌幅')
            
            available_cols = [c for c in other_display_cols if c in other_stocks.columns]
            print(other_stocks[available_cols].to_string(index=False))
            
            print(f"\n💡 无法处理的原因:")
            for idx, row in other_stocks.iterrows():
                code = str(row['代码'])
                code_len = len(code)
                code_prefix = code[:2] if code_len >= 2 else ""
                print(f"   • {row['代码']} {row.get('名称', 'N/A')}: 长度={code_len}, 前缀={code_prefix}（代码格式不支持）")
        
        print(f"\n" + "=" * 80)
        print(f"💡 数据说明:")
        print(f"   • 涨停家数：{limit_up_count}只（包含所有交易所）")
        print(f"   • JQData支持：{len(jqdata_stocks)}只（沪深A股，用于JQData查询）")
        if len(bse_stocks) > 0:
            print(f"   • 北交所股票：{len(bse_stocks)}只（使用AKShare数据补充）")
        print(f"   • 后续分析：")
        print(f"     - 情绪周期判断：使用全部涨停家数（{limit_up_count}只）")
        print(f"     - 连板高度计算：JQData股票使用JQData，北交所股票使用AKShare")
        print("=" * 80)
        
        # 显示统计信息（使用全部数据）
        print(f"\n📊 涨停板统计信息（全部股票）:")
        print(f"   平均涨跌幅: {limit_up_data['涨跌幅'].mean():.2f}%")
        print(f"   平均换手率: {limit_up_data['换手率'].mean():.2f}%")
        if '封板资金' in limit_up_data.columns:
            print(f"   总封板资金: {limit_up_data['封板资金'].sum()/100000000:.2f}亿元")
        
        print(f"\n涨停股票列表（前60只，包含所有交易所）:")
        display_cols = ['代码', '名称', '最新价', '涨跌幅', 'exchange_type']
        if '换手率' in limit_up_data.columns:
            display_cols.append('换手率')
        if '封板资金' in limit_up_data.columns:
            display_cols.append('封板资金')
        if '首次封板时间' in limit_up_data.columns:
            display_cols.append('首次封板时间')
        
        # 重命名exchange_type为更友好的显示
        display_data = limit_up_data.copy()
        display_data['交易所'] = display_data['exchange_type'].map({
            'XSHE': '深市',
            'XSHG': '沪市',
            'BSE': '北交所',
            'OTHER': '其他'
        })
        display_cols = [c if c != 'exchange_type' else '交易所' for c in display_cols]
        
        available_cols = [c for c in display_cols if c in display_data.columns]
        print(display_data[available_cols].head(60).to_string(index=False))
        
        # 保留所有数据，不过滤
        # limit_up_data 已包含所有股票（包括北交所）
    else:
        limit_up_count = 0
        print("⚠️  今日暂无涨停股票")
        limit_up_data = None
except Exception as e:
    print(f"❌ 获取失败: {e}")
    import traceback
    traceback.print_exc()
    limit_up_count = 0
    limit_up_data = None

📅 当前日期: 2026-01-14

📊 正在获取涨停板数据...
✅ 获取成功！涨停家数: 102只
✅ JQData支持: 99只（沪深A股）
📊 北交所股票: 3只（将使用AKShare数据补充）

📋 北交所股票列表（共 3 只）
    代码   名称    最新价       涨跌幅       换手率  连板数
920227 美登科技 115.76 29.994387 32.663357    1
920169 七丰精工  46.15 30.000002 41.095253    1
920021 流金科技  13.57 29.980844 54.367287    1

💡 北交所股票说明:
   • 代码格式：92开头（北京证券交易所）
   • JQData支持：❌ 不支持（JQData目前不支持北交所股票）
   • 数据补充：✅ 将使用AKShare历史数据计算连板高度
   • 影响分析：北交所股票通常市值较小，对整体市场情绪影响有限

💡 数据说明:
   • 涨停家数：102只（包含所有交易所）
   • JQData支持：99只（沪深A股，用于JQData查询）
   • 北交所股票：3只（使用AKShare数据补充）
   • 后续分析：
     - 情绪周期判断：使用全部涨停家数（102只）
     - 连板高度计算：JQData股票使用JQData，北交所股票使用AKShare

📊 涨停板统计信息（全部股票）:
   平均涨跌幅: 12.46%
   平均换手率: 16.31%
   总封板资金: 142.55亿元

涨停股票列表（前60只，包含所有交易所）:
    代码    名称    最新价       涨跌幅 交易所       换手率       封板资金 首次封板时间
002044  美年健康   8.15  9.986505  深市  8.257297  681189982 092500
002112  三变科技  19.23 10.011442  深市  4.667266  263850003 092500
002115  三维通信  19.59  9.994386  深市  1.010550 1570949232 092500
002153  石基信息  14.67  9.970016  深市  1.

## 📊 2. 计算连板高度

In [20]:
if limit_up_data is not None and not limit_up_data.empty and jq:
    print("📊 正在计算连板高度...")
    
    # 确保today_display已定义（从Cell 7获取或使用当前日期）
    try:
        if 'today_display' not in globals():
            from datetime import datetime
            today_display = datetime.now().strftime('%Y-%m-%d')
            print(f"  ℹ️  使用当前日期: {today_display}")
        else:
            print(f"  ℹ️  使用已定义日期: {today_display}")
    except:
        from datetime import datetime
        today_display = datetime.now().strftime('%Y-%m-%d')
        print(f"  ℹ️  使用当前日期: {today_display}")
    
    def calculate_consecutive_limit_up(jq_code, end_date, days=10):
        """
        计算连续涨停天数（修复版）
        
        参数:
            jq_code: JQData格式的股票代码
            end_date: 结束日期（格式：YYYY-MM-DD）
            days: 查询天数
        
        逻辑：
        1. 从最新日期往前倒推
        2. 如果收盘价 >= 涨停价 * 0.995，认为是涨停
        3. 连续计算直到不是涨停为止
        """
        if not jq_code:
            return 0
        
        try:
            # 使用JQDataClient的get_price_by_count方法（支持count参数）
            if hasattr(jq, 'get_price_by_count'):
                # JQDataClient对象
                price_data = jq.get_price_by_count(
                    security=jq_code,
                    count=days,
                    end_date=end_date,
                    frequency='daily',
                    fields=['close', 'high_limit']
                )
            else:
                # 直接使用jqdatasdk
                import jqdatasdk as jq_sdk
                price_data = jq_sdk.get_price(
                    jq_code,
                    count=days,
                    end_date=end_date,
                    frequency='daily',
                    fields=['close', 'high_limit']
                )
            
            if price_data is None or price_data.empty:
                return 0
            
            consecutive = 0
            for i in range(len(price_data)-1, -1, -1):
                close = price_data['close'].iloc[i]
                high_limit = price_data['high_limit'].iloc[i]
                
                # 判断是否涨停：收盘价接近涨停价（允许0.5%误差）
                if high_limit > 0 and close >= high_limit * 0.995:
                    consecutive += 1
                else:
                    break
            
            return consecutive
        except Exception as e:
            # 添加调试信息：前3个错误打印出来
            if not hasattr(calculate_consecutive_limit_up, '_error_count'):
                calculate_consecutive_limit_up._error_count = 0
            if calculate_consecutive_limit_up._error_count < 3:
                print(f"  ⚠️  计算连板错误 ({jq_code}): {str(e)[:80]}")
                calculate_consecutive_limit_up._error_count += 1
            return 0
    
    # 计算每只涨停股票的连板高度（处理所有有效代码的股票）
    # 只处理有jq_code的股票
    valid_stocks = limit_up_data[limit_up_data['jq_code'].notna()].copy()
    print(f"  处理 {len(valid_stocks)} 只有效代码的股票...")
    
    # 初始化连板高度列
    if '连板高度' not in limit_up_data.columns:
        limit_up_data['连板高度'] = 0
    
    processed = 0
    success_count = 0
    for idx, row in valid_stocks.iterrows():
        if processed % 10 == 0 and processed > 0:
            print(f"  处理进度: {processed}/{len(valid_stocks)}")
        jq_code = row.get('jq_code')
        if jq_code:
            height = calculate_consecutive_limit_up(jq_code, today_display, days=10)
            limit_up_data.loc[idx, '连板高度'] = height
            if height > 0:
                success_count += 1
        processed += 1
    
    print(f"  ✅ 处理完成，找到 {success_count} 只有连板的股票")
    
    # 统计连板高度分布
    height_data = limit_up_data[limit_up_data['连板高度'] > 0]
    if not height_data.empty:
        height_dist = height_data['连板高度'].value_counts().sort_index()
        max_height = height_data['连板高度'].max()
        
        print(f"\n✅ 连板高度统计:")
        print(height_dist.to_string())
        print(f"\n📊 最高连板: {max_height}板")
        print(f"📊 连板股票总数: {len(height_data)}只")
        
        # 显示所有连板股票详情（包括1板）
        all_board_stocks = height_data.sort_values('连板高度', ascending=False)
        print(f"\n📋 所有连板股票列表（{len(all_board_stocks)}只，按连板高度排序）:")
        display_cols = ['代码', '名称', '最新价', '涨跌幅', '连板高度']
        if '换手率' in all_board_stocks.columns:
            display_cols.insert(4, '换手率')
        print(all_board_stocks[display_cols].to_string(index=False))
        
        # 单独显示2板以上的股票
        high_board_stocks = height_data[height_data['连板高度'] >= 2].sort_values('连板高度', ascending=False)
        if not high_board_stocks.empty:
            print(f"\n🔥 2板以上股票（{len(high_board_stocks)}只）:")
            print(high_board_stocks[display_cols].to_string(index=False))
    else:
        print("\n⚠️  无连板数据")
        print("   可能原因：")
        print("   1. 今日涨停股票都是首板（1板）")
        print("   2. 数据获取或计算出现问题")
        max_height = 0
else:
    max_height = 0
    if not jq:
        print("⚠️  无法计算连板高度（需要JQData连接）")
    else:
        print("⚠️  无法计算连板高度（需要涨停板数据）")

📊 正在计算连板高度...
  ℹ️  使用已定义日期: 2026-01-14
  处理 99 只JQData支持的股票（沪深A股）...


2026-01-14 07:45:08,657 - jqdata.client - INFO - 获取价格数据成功: 002044.XSHE, 10条
2026-01-14 07:45:08,905 - jqdata.client - INFO - 获取价格数据成功: 002112.XSHE, 10条
2026-01-14 07:45:09,153 - jqdata.client - INFO - 获取价格数据成功: 002115.XSHE, 10条
2026-01-14 07:45:09,384 - jqdata.client - INFO - 获取价格数据成功: 002153.XSHE, 10条
2026-01-14 07:45:09,627 - jqdata.client - INFO - 获取价格数据成功: 002219.XSHE, 10条
2026-01-14 07:45:10,424 - jqdata.client - INFO - 获取价格数据成功: 002465.XSHE, 10条
2026-01-14 07:45:10,667 - jqdata.client - INFO - 获取价格数据成功: 002879.XSHE, 10条
2026-01-14 07:45:10,923 - jqdata.client - INFO - 获取价格数据成功: 601992.XSHG, 10条
2026-01-14 07:45:11,163 - jqdata.client - INFO - 获取价格数据成功: 601116.XSHG, 10条
2026-01-14 07:45:11,391 - jqdata.client - INFO - 获取价格数据成功: 603056.XSHG, 10条


  处理进度: 10/99


2026-01-14 07:45:12,083 - jqdata.client - INFO - 获取价格数据成功: 000409.XSHE, 10条
2026-01-14 07:45:12,329 - jqdata.client - INFO - 获取价格数据成功: 600986.XSHG, 10条
2026-01-14 07:45:12,559 - jqdata.client - INFO - 获取价格数据成功: 002131.XSHE, 10条
2026-01-14 07:45:12,807 - jqdata.client - INFO - 获取价格数据成功: 603179.XSHG, 10条
2026-01-14 07:45:13,036 - jqdata.client - INFO - 获取价格数据成功: 002429.XSHE, 10条
2026-01-14 07:45:13,264 - jqdata.client - INFO - 获取价格数据成功: 002264.XSHE, 10条
2026-01-14 07:45:13,520 - jqdata.client - INFO - 获取价格数据成功: 002969.XSHE, 10条
2026-01-14 07:45:13,749 - jqdata.client - INFO - 获取价格数据成功: 003007.XSHE, 10条
2026-01-14 07:45:14,431 - jqdata.client - INFO - 获取价格数据成功: 605118.XSHG, 10条
2026-01-14 07:45:14,659 - jqdata.client - INFO - 获取价格数据成功: 600785.XSHG, 10条


  处理进度: 20/99


...

  处理进度: 80/99


2026-01-14 07:45:32,796 - jqdata.client - INFO - 获取价格数据成功: 002004.XSHE, 10条
2026-01-14 07:45:33,103 - jqdata.client - INFO - 获取价格数据成功: 603829.XSHG, 10条
2026-01-14 07:45:33,392 - jqdata.client - INFO - 获取价格数据成功: 688080.XSHG, 10条
2026-01-14 07:45:33,619 - jqdata.client - INFO - 获取价格数据成功: 600589.XSHG, 10条
2026-01-14 07:45:33,884 - jqdata.client - INFO - 获取价格数据成功: 600133.XSHG, 10条
2026-01-14 07:45:34,127 - jqdata.client - INFO - 获取价格数据成功: 301396.XSHE, 10条
2026-01-14 07:45:34,513 - jqdata.client - INFO - 获取价格数据成功: 002353.XSHE, 10条
2026-01-14 07:45:34,946 - jqdata.client - INFO - 获取价格数据成功: 600410.XSHG, 10条
2026-01-14 07:45:35,253 - jqdata.client - INFO - 获取价格数据成功: 002281.XSHE, 10条
2026-01-14 07:45:35,580 - jqdata.client - INFO - 获取价格数据成功: 603353.XSHG, 10条


  处理进度: 90/99


2026-01-14 07:45:35,869 - jqdata.client - INFO - 获取价格数据成功: 002290.XSHE, 10条
2026-01-14 07:45:36,687 - jqdata.client - INFO - 获取价格数据成功: 603988.XSHG, 10条
2026-01-14 07:45:36,994 - jqdata.client - INFO - 获取价格数据成功: 603280.XSHG, 10条
2026-01-14 07:45:37,301 - jqdata.client - INFO - 获取价格数据成功: 605287.XSHG, 10条
2026-01-14 07:45:37,586 - jqdata.client - INFO - 获取价格数据成功: 605255.XSHG, 10条
2026-01-14 07:45:37,916 - jqdata.client - INFO - 获取价格数据成功: 688292.XSHG, 10条
2026-01-14 07:45:38,222 - jqdata.client - INFO - 获取价格数据成功: 000078.XSHE, 10条
2026-01-14 07:45:38,520 - jqdata.client - INFO - 获取价格数据成功: 300792.XSHE, 10条
2026-01-14 07:45:38,837 - jqdata.client - INFO - 获取价格数据成功: 603687.XSHG, 10条


  ✅ JQData股票处理完成，找到 99 只有连板的股票

  处理 3 只北交所股票（使用AKShare）...
  ✅ 北交所股票处理完成，找到 3 只有连板的股票

  ✅ 全部处理完成，共找到 102 只有连板的股票（包含所有交易所）

✅ 连板高度统计:
连板高度
1    79
2     5
3    13
4     4
5     1

📊 最高连板: 5板
📊 连板股票总数: 102只

📋 所有连板股票列表（102只，按连板高度排序）:
    代码    名称    最新价       涨跌幅       换手率  连板高度
003007  直真科技  59.77  9.992639 35.193939     5
002044  美年健康   8.15  9.986505  8.257297     4
002131  利欧股份   9.93  9.966777 10.463186     4
002115  三维通信  19.59  9.994386  1.010550     4
002400  省广集团  13.81 10.039841 24.508476     4
002465  海格通信  26.62 10.000000 25.912104     3
002718  友邦吊顶  68.56  9.995187 14.042332     3
601116  三江购物  20.88 10.010537  1.128210     3
603888   新华网  29.60  9.996284 11.688195     3
601992  金隅集团   2.34  9.859155  8.586209     3
001255  博菲电气  45.63 10.004822 12.969127     3
603000   人民网  28.03 10.007850  9.921249     3
920227  美登科技 115.76 29.994387 32.663357     3
603056  德邦股份  15.44  9.971510  0.236213     3
000593  德龙汇能  17.41  9.981049 20.331432     3
301378   通达海  54.59 20.004396 

## 📊 3. 计算炸板率

## 📊 3.1 盘中走势分析（涨停-开板-再涨停）

**功能**：分析特定股票的盘中走势，识别"涨停-开板-再涨停"模式

**分析方法**：
1. 获取分时数据（1分钟级别）
2. 识别涨停时间段和开板时间段
3. 计算回调幅度和持续时间
4. 判断走势模式和市场含义

**使用场景**：
- 分析龙头股票的盘中表现
- 判断封板稳定性
- 评估资金博弈情况

In [21]:
"""
盘中走势分析：涨停-开板-再涨停模式
分析特定股票的盘中走势，识别关键时间点和走势模式
"""

from core.stock_intraday_analyzer import StockIntradayAnalyzer
from datetime import datetime, timezone, timedelta

def cn_today_str():
    """获取中国时间（UTC+8）的当前日期字符串"""
    cn_now = datetime.now(timezone.utc) + timedelta(hours=8)
    return cn_now.strftime('%Y-%m-%d')

# 要分析的股票（可以修改）
target_code = "002400"  # 省广集团
target_name = "省广集团"
analysis_date = cn_today_str()

print("=" * 80)
print(f"📊 盘中走势分析：{target_name}（{target_code}）")
print("=" * 80)
print(f"分析日期: {analysis_date}")

# 创建分析器
analyzer = StockIntradayAnalyzer()

# 分析走势模式
result = analyzer.analyze_limit_up_pattern(target_code, analysis_date)

if result.get('success'):
    print(f"\n✅ 分析成功！")
    print(f"\n📊 走势模式: {result['pattern']}")
    print(f"   涨停次数: {result['limit_up_count']}次")
    print(f"   开板次数: {result['open_count']}次")
    print(f"   最大回调: {result['max_drawdown_pct']:.2f}%")
    
    if result.get('first_limit_up_time'):
        print(f"   首次涨停: {result['first_limit_up_time']}")
    if result.get('last_limit_up_time'):
        print(f"   最后涨停: {result['last_limit_up_time']}")
    
    # 显示涨停时间段详情
    if result.get('limit_up_periods'):
        print(f"\n📈 涨停时间段详情:")
        for period in result['limit_up_periods']:
            print(f"   第{period['period']}次涨停:")
            print(f"     时间: {period['start_time']} 至 {period['end_time']}")
            print(f"     持续: {period['duration_minutes']:.0f}分钟")
            print(f"     价格: {period['start_price']:.2f}元 → {period['end_price']:.2f}元")
            print(f"     成交量: {period['total_volume']:,}手")
    
    # 显示开板时间段详情
    if result.get('open_periods'):
        print(f"\n📉 开板时间段详情:")
        for period in result['open_periods']:
            print(f"   第{period['period']}次开板:")
            print(f"     时间: {period['start_time']} 至 {period['end_time']}")
            print(f"     持续: {period['duration_minutes']:.0f}分钟")
            print(f"     价格: {period['start_price']:.2f}元 → {period['end_price']:.2f}元")
            print(f"     最低: {period['min_price']:.2f}元")
            print(f"     回调幅度: {period['drawdown_pct']:.2f}%")
            print(f"     成交量: {period['total_volume']:,}手")
    
    # 显示走势解释
    print(f"\n" + "=" * 80)
    print(f"💡 走势分析")
    print("=" * 80)
    interpretation = analyzer.interpret_pattern(result)
    print(interpretation)
    
    # 保存结果供后续使用
    intraday_analysis_result = result
    
else:
    print(f"\n❌ 分析失败: {result.get('error', '未知错误')}")
    intraday_analysis_result = None

print("\n" + "=" * 80)

📊 盘中走势分析：省广集团（002400）
分析日期: 2026-01-14

✅ 分析成功！

📊 走势模式: open_then_limit_up
   涨停次数: 1次
   开板次数: 1次
   最大回调: 2.48%
   首次涨停: 2026-01-14 09:53:00
   最后涨停: 2026-01-14 09:53:00

📈 涨停时间段详情:
   第1次涨停:
     时间: 2026-01-14 09:53:00 至 2026-01-14 15:00:00
     持续: 307分钟
     价格: 13.81元 → 13.81元
     成交量: 1,106,648手

📉 开板时间段详情:
   第1次开板:
     时间: 2026-01-14 09:30:00 至 2026-01-14 09:52:00
     持续: 22分钟
     价格: 12.50元 → 13.72元
     最低: 12.19元
     回调幅度: 2.48%
     成交量: 3,123,921手

💡 走势分析

✅ **开板后涨停模式**

📊 特征：
   • 盘中开板，随后涨停

💡 市场含义：
   1. **洗盘充分**：开板期间完成换手
   2. **资金认可**：开板后资金继续买入
   3. **封板较稳**：开板后涨停，说明买盘力量强

🎯 操作建议：
   • 封板较稳，可考虑持有
   • 关注开板期间的成交量
   • 如果开板时间短且回调小，封板更稳
            



In [39]:
if limit_up_data is not None and not limit_up_data.empty:
    print("📊 正在计算炸板率...")
    
    zhaban_rate = 0
    zhaban_count = 0
    total_limit_up_attempts = limit_up_count
    zhaban_stocks = None
    
    # 获取涨停股票代码集合（确保为字符串类型）
    limit_up_codes = set(limit_up_data['代码'].astype(str).values) if limit_up_data is not None else set()
    
    # 方法1: 尝试使用AKShare获取实时行情（带重试和超时处理）
    success = False
    max_retries = 3
    timeout_seconds = 30
    
    for attempt in range(max_retries):
        try:
            print(f"  尝试获取实时行情数据（第{attempt+1}/{max_retries}次，超时={timeout_seconds}秒）...")
            
            # 设置超时
            import socket
            original_timeout = socket.getdefaulttimeout()
            socket.setdefaulttimeout(timeout_seconds)
            
            try:
                # 使用AKShare获取所有股票实时行情
                all_stocks = ak.stock_zh_a_spot_em()
                
                if all_stocks is not None and not all_stocks.empty:
                    # 确保代码列为字符串类型
                    all_stocks['代码'] = all_stocks['代码'].astype(str)
                    
                    # 筛选炸板股票（涨跌幅>=9%但<9.5%，且不是涨停板数据中的）
                    # 注意：北交所股票涨跌幅限制是30%，但这里使用9.5%作为通用标准
                    zhaban_stocks = all_stocks[
                        (all_stocks['涨跌幅'] >= 9.0) & 
                        (all_stocks['涨跌幅'] < 9.5) & 
                        (~all_stocks['代码'].isin(limit_up_codes))
                    ].copy()
                    
                    zhaban_count = len(zhaban_stocks)
                    total_limit_up_attempts = limit_up_count + zhaban_count
                    
                    if total_limit_up_attempts > 0:
                        zhaban_rate = zhaban_count / total_limit_up_attempts * 100
                    
                    success = True
                    print(f"  ✅ 获取成功！共 {len(all_stocks)} 只股票，找到 {zhaban_count} 只炸板股票")
                    break
                else:
                    print(f"  ⚠️  返回数据为空")
            finally:
                # 恢复原始超时设置
                socket.setdefaulttimeout(original_timeout)
                
        except Exception as e:
            error_msg = str(e)
            if "timeout" in error_msg.lower() or "timed out" in error_msg.lower():
                print(f"  ⚠️  网络超时（第{attempt+1}次）: {error_msg[:100]}")
                if attempt < max_retries - 1:
                    timeout_seconds += 10  # 增加超时时间
                    print(f"    下次尝试将使用更长的超时时间: {timeout_seconds}秒")
            else:
                print(f"  ⚠️  获取失败（第{attempt+1}次）: {error_msg[:100]}")
            
            if attempt == max_retries - 1:
                print(f"  ❌ 所有尝试均失败，使用降级方案")
    
    # 如果方法1失败，尝试降级方案
    if not success:
        print(f"\n  💡 尝试降级方案：使用估算值")
        try:
            # 方法2: 使用历史平均值估算
            # 根据市场经验，炸板率通常在5-15%之间
            # 这里使用保守估计：假设炸板数量为涨停数量的5%
            estimated_zhaban_ratio = 0.05  # 5%
            zhaban_count = max(1, int(limit_up_count * estimated_zhaban_ratio))
            total_limit_up_attempts = limit_up_count + zhaban_count
            zhaban_rate = zhaban_count / total_limit_up_attempts * 100 if total_limit_up_attempts > 0 else 0
            
            print(f"  ⚠️  使用估算值：假设炸板率为{estimated_zhaban_ratio*100:.0f}%")
            print(f"  💡 说明：由于无法获取实时行情数据，使用历史平均值估算")
            print(f"  ⚠️  注意：这是估算值，实际炸板率可能不同")
            
        except Exception as e2:
            print(f"  ❌ 降级方案也失败: {str(e2)[:100]}")
            zhaban_rate = 0
            zhaban_count = 0
    
    # 显示结果
    print(f"\n" + "=" * 80)
    print(f"📊 炸板率统计结果")
    print("=" * 80)
    print(f"  涨停成功: {limit_up_count}只")
    print(f"  炸板数量: {zhaban_count}只" + ("（估算）" if not success else ""))
    print(f"  总尝试数: {total_limit_up_attempts}只")
    print(f"  炸板率: {zhaban_rate:.2f}%" + ("（估算值，仅供参考）" if not success else ""))
    
    if success and zhaban_count > 0:
        print(f"\n📋 炸板股票列表（前10只）:")
        display_cols = ['代码', '名称', '最新价', '涨跌幅']
        if '换手率' in zhaban_stocks.columns:
            display_cols.append('换手率')
        if '成交量' in zhaban_stocks.columns:
            display_cols.append('成交量')
        available_cols = [c for c in display_cols if c in zhaban_stocks.columns]
        print(zhaban_stocks[available_cols].head(10).to_string(index=False))
    elif not success:
        print(f"\n💡 建议:")
        print(f"   • 检查网络连接")
        print(f"   • 稍后重试（可能是数据源暂时不可用）")
        print(f"   • 或手动查看实时行情数据")
    
    print("=" * 80)
else:
    zhaban_rate = 0
    zhaban_count = 0
    print("⚠️  无法计算炸板率（需要涨停板数据）")

📊 正在计算炸板率...
  📅 统计日期: 20260114
  ✅ 获取成功！数据来源: akshare
     炸板数量: 59只
     涨停成功: 102只
     总尝试数: 161只
     炸板率: 36.65%

📊 炸板率统计结果
  涨停成功: 102只
  炸板数量: 59只
  总尝试数: 161只
  炸板率: 36.65%

📊 炸板率风险评估:
  🔴 炸板率过高（36.65% > 30%），市场情绪不稳，建议谨慎
  尝试获取炸板过程数据（第1/3次）...
  ✅ 获取成功！找到 59 只炸板股票

📊 炸板率统计结果
  涨停成功: 102只
  炸板数量: 59只
  总尝试数: 161只
  炸板率: 36.65%

📋 炸板股票列表（前10只）:
    代码   名称    最新价       涨跌幅       换手率  炸板次数
600676 交运股份   9.03  9.854015 14.969301    49
603739 蔚蓝生物  16.71 -0.179211 16.580118     1
300816  艾可蓝  80.07 16.043478 32.892506     4
002223 鱼跃医疗  45.40  7.838480  5.284490     3
605288 凯迪股份 115.80  5.976023  2.790983     1
002842 翔鹭钨业  17.40  9.090909 27.671047     1
002769  普路通  14.24  3.563637 21.218874     1
000969 安泰科技  29.27  3.464122 20.377483     4
002354 天娱数科   8.50  9.819121 34.262245     5
603285 键邦股份  27.35  7.889546 22.271547     1

📊 炸板率风险评估:
  🔴 炸板率过高（36.65% > 30%），市场情绪不稳，建议谨慎


## 📊 4. 分析资金流向

In [23]:
# 🧪 资金流向数据获取整合测试（完整版）
# 整合方案：聚宽+AKShare，降级策略
# 包含：大盘资金流向、北向资金、行业资金流向

import akshare as ak
import pandas as pd
import numpy as np
from datetime import datetime, timezone, timedelta

print("=" * 80)
print("🧪 资金流向数据获取整合测试（完整版）")
print(f"当前时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

# ========== 辅助函数：日期对齐和有效性判定 ==========
def cn_today_str():
    """获取中国时间（UTC+8）的当前日期字符串"""
    cn_now = datetime.now(timezone.utc) + timedelta(hours=8)
    return cn_now.strftime('%Y-%m-%d')

def is_valid_north(date_str, total_net, min_abs_value=0.01):
    """
    判定北向资金数据是否有效
    
    参数:
        date_str: 交易日字符串
        total_net: 净买额（亿元）
        min_abs_value: 最小绝对值阈值（亿元）
    
    返回:
        bool: 是否有效
    """
    # 1) 未来日期无效（相对中国当天，严格大于才无效）
    cn_today = cn_today_str()
    date_str_clean = str(date_str).strip()
    if date_str_clean > cn_today:
        return False
    
    # 2) 0值（或接近0）无效，触发降级
    # 注意：非交易时段可能返回0，应降级到历史数据
    if abs(float(total_net)) < min_abs_value:
        return False
    
    return True

results = {}

# ========== 1. 大盘资金流向 ==========
print("\n" + "=" * 80)
print("📊 1. 大盘资金流向")
print("=" * 80)

# 获取目标日期（中国时区）
target_date_str = cn_today_str()
target_date = pd.to_datetime(target_date_str)

try:
    market_flow = ak.stock_market_fund_flow()
    if market_flow is not None and not market_flow.empty:
        # 转换日期列为datetime类型
        market_flow['日期'] = pd.to_datetime(market_flow['日期'])
        
        # 按日期排序（最新的在前）
        market_flow_sorted = market_flow.sort_values('日期', ascending=False)
        
        # 显示数据范围
        min_date = market_flow_sorted['日期'].min()
        max_date = market_flow_sorted['日期'].max()
        print(f"📅 数据范围: {min_date.strftime('%Y-%m-%d')} 至 {max_date.strftime('%Y-%m-%d')} (共{len(market_flow_sorted)}个交易日)")
        
        # 尝试获取目标日期的数据
        target_data = market_flow_sorted[market_flow_sorted['日期'] == target_date]
        
        if not target_data.empty:
            # 找到目标日期的数据
            data = target_data.iloc[0]
            print(f"✅ 找到目标日期数据: {target_date_str}")
        else:
            # 如果目标日期不在数据中，使用最新数据
            data = market_flow_sorted.iloc[0]
            data_date = data['日期'].strftime('%Y-%m-%d')
            print(f"⚠️  目标日期 {target_date_str} 不在数据列表中")
            print(f"   使用最新数据: {data_date}")
            if target_date > max_date:
                days_diff = (target_date - max_date).days
                print(f"   💡 目标日期比最新数据晚 {days_diff} 天（可能是未来日期，等待数据更新）")
            elif target_date < min_date:
                days_diff = (min_date - target_date).days
                print(f"   💡 目标日期比最早数据早 {days_diff} 天（数据已过期）")
        
        # 单位处理：AKShare返回的是元，需要除以1e8转换为亿元
        raw_amount = float(data['主力净流入-净额'])
        amount_yi = raw_amount / 1e8  # 转换为亿元
        
        print(f"\n📊 大盘资金流向数据:")
        print(f"   日期: {data['日期'].strftime('%Y-%m-%d')}")
        print(f"   上证涨跌幅: {data['上证-涨跌幅']:.2f}%")
        print(f"   深证涨跌幅: {data['深证-涨跌幅']:.2f}%")
        print(f"   主力净流入: {amount_yi:.2f}亿元 ({data['主力净流入-净占比']:.2f}%)")
        print(f"   超大单净流入: {data['超大单净流入-净额'] / 1e8:.2f}亿元 ({data['超大单净流入-净占比']:.2f}%)")
        print(f"   大单净流入: {data['大单净流入-净额'] / 1e8:.2f}亿元 ({data['大单净流入-净占比']:.2f}%)")
        print(f"   中单净流入: {data['中单净流入-净额'] / 1e8:.2f}亿元 ({data['中单净流入-净占比']:.2f}%)")
        print(f"   小单净流入: {data['小单净流入-净额'] / 1e8:.2f}亿元 ({data['小单净流入-净占比']:.2f}%)")
        
        results['大盘'] = {
            '日期': data['日期'].strftime('%Y-%m-%d'),
            '主力净流入(亿)': amount_yi,
            '主力净流入(%)': float(data['主力净流入-净占比']),
            '超大单净流入(亿)': data['超大单净流入-净额'] / 1e8,
            '大单净流入(亿)': data['大单净流入-净额'] / 1e8,
            '数据源': 'AKShare stock_market_fund_flow'
        }
    else:
        print("⚠️  返回数据为空")
        results['大盘'] = {
            '日期': target_date_str,
            '主力净流入(亿)': 0.0,
            '主力净流入(%)': 0.0,
            '数据源': '无数据'
        }
except Exception as e:
    print(f"❌ 获取失败: {str(e)[:150]}")
    import traceback
    traceback.print_exc()
    results['大盘'] = {
        '日期': target_date_str,
        '主力净流入(亿)': 0.0,
        '主力净流入(%)': 0.0,
        '数据源': '获取失败'
    }

# ========== 2. 北向资金（暂时注释，待数据源稳定后启用） ==========
# 注释原因：
# 1. 数据源不稳定（汇总数据可能返回未来日期+0值）
# 2. 历史数据只到2024-08-16，无法获取最新数据
# 3. 根据第一性原理，陈小群战法更关注：涨停家数、连板高度、炸板率、大盘主力、行业资金
# 4. 北向资金作为辅助指标，在当前数据源不稳定时暂不使用

# print("\n" + "=" * 80)
# print("📊 2. 北向资金（暂时注释）")
# print("=" * 80)
# print("  ⚠️  北向资金数据源暂时不稳定，已注释")
# print("  根据第一性原理，陈小群战法核心指标：")
# print("  1. 涨停家数（赚钱效应，最重要）")
# print("  2. 连板高度（情绪强度验证）")
# print("  3. 炸板率（风险信号）")
# print("  4. 大盘主力净流入（整体资金态度）")
# print("  5. 行业资金净流入（结构性机会）")

# ========== 2. 行业资金流向 ==========
print("\n" + "=" * 80)
print("📊 2. 行业资金流向（结构性机会指标）")
print("=" * 80)
print(f"💡 说明: '今日'指最新交易日，如果目标日期是最新交易日，可以获取")
print(f"   目标日期: {target_date_str}")

sector_success = False

# 2.1 先尝试今日数据（最新交易日）
print(f"\n  2.1 尝试'今日'数据（最新交易日）")
try:
    sector_today = ak.stock_sector_fund_flow_rank(indicator="今日", sector_type="行业资金流")
    if sector_today is not None and not sector_today.empty:
        # 筛选有效数据（非空值）
        valid_data = sector_today[sector_today['今日主力净流入-净额'].notna()]
        
        if len(valid_data) > 0:
            # 单位处理：AKShare返回的是元，需要除以1e8转换为亿元
            total_inflow = valid_data['今日主力净流入-净额'].sum() / 1e8
            top3 = valid_data.nlargest(3, '今日主力净流入-净额')
            
            print(f"  ✅ '今日'数据有效，行业数: {len(valid_data)}")
            print(f"     行业总主力净流入: {total_inflow:.2f}亿元")
            print(f"     主力流入前3: {', '.join(top3['名称'].tolist())}")
            
            # 显示前3名详细信息
            print(f"\n     前3名行业详情:")
            for idx, row in top3.iterrows():
                inflow_yi = row['今日主力净流入-净额'] / 1e8
                pct = row['今日主力净流入-净占比']
                print(f"       {row['名称']}: {inflow_yi:.2f}亿元 ({pct:.2f}%)")
            
            results['行业'] = {
                '总净流入(亿)': total_inflow,
                '行业数': len(valid_data),
                '前3行业': top3['名称'].tolist(),
                '数据源': '今日（最新交易日）',
                '日期': target_date_str  # 假设是最新交易日
            }
            sector_success = True
        else:
            print(f"  ⚠️  '今日'数据为空（所有行业数据为NaN）")
    else:
        print(f"  ⚠️  '今日'数据返回为空")
except Exception as e:
    print(f"  ❌ '今日'数据获取失败: {str(e)[:100]}")

# 2.2 如果今日数据为空，使用5日数据作为降级方案
if not sector_success:
    print(f"\n  2.2 尝试'5日'数据作为降级方案")
    try:
        sector_5d = ak.stock_sector_fund_flow_rank(indicator="5日", sector_type="行业资金流")
        if sector_5d is not None and not sector_5d.empty:
            valid_data = sector_5d[sector_5d['5日主力净流入-净额'].notna()]
            if len(valid_data) > 0:
                total_inflow = valid_data['5日主力净流入-净额'].sum() / 1e8
                top3 = valid_data.nlargest(3, '5日主力净流入-净额')
                
                print(f"  ✅ '5日'数据有效，行业数: {len(valid_data)}")
                print(f"     ⚠️  注意：这是5日汇总数据，不是单日数据")
                print(f"     5日行业总主力净流入: {total_inflow:.2f}亿元")
                print(f"     5日主力流入前3: {', '.join(top3['名称'].tolist())}")
                
                results['行业'] = {
                    '总净流入(亿)': total_inflow,
                    '行业数': len(valid_data),
                    '前3行业': top3['名称'].tolist(),
                    '数据源': '5日（降级方案）',
                    '日期': target_date_str,
                    '说明': '5日汇总数据，非单日数据'
                }
                sector_success = True
            else:
                print(f"  ⚠️  '5日'数据为空")
        else:
            print(f"  ⚠️  '5日'数据返回为空")
    except Exception as e:
        print(f"  ❌ '5日'数据获取失败: {str(e)[:100]}")

# 2.3 如果都失败，设置默认值
if not sector_success:
    print(f"\n  ⚠️  所有数据源获取失败，设置默认值")
    results['行业'] = {
        '总净流入(亿)': 0.0,
        '行业数': 0,
        '前3行业': [],
        '数据源': '获取失败',
        '日期': target_date_str
    }

# ========== 汇总结果 ==========
print("\n" + "=" * 80)
print("📈 数据获取汇总")
print("=" * 80)

for key, value in results.items():
    print(f"\n{key}:")
    for k, v in value.items():
        if isinstance(v, float):
            if '亿' in k or '净流入' in k:
                print(f"  {k}: {v:.2f}")
            elif '%' in k or '占比' in k:
                print(f"  {k}: {v:.2f}")
            else:
                print(f"  {k}: {v:.2f}")
        elif isinstance(v, list):
            if len(v) > 0:
                print(f"  {k}: {', '.join(v)}")
            else:
                print(f"  {k}: []")
        else:
            print(f"  {k}: {v}")

# 显示数据源信息
print(f"\n💡 数据源说明:")
if '大盘' in results:
    print(f"   大盘资金流向: {results['大盘'].get('数据源', '未知')}")
if '行业' in results:
    print(f"   行业资金流向: {results['行业'].get('数据源', '未知')}")
    if '说明' in results['行业']:
        print(f"   ⚠️  注意: {results['行业']['说明']}")

# ========== 综合情绪判断（基于第一性原理） ==========
print("\n" + "=" * 80)
print("🎯 综合情绪判断（基于第一性原理）")
print("=" * 80)
print("\n💡 第一性原理分析：")
print("   市场情绪 = 赚钱效应 + 资金态度 + 风险信号")
print("   - 赚钱效应 = 涨停家数（最重要）+ 连板高度（验证）")
print("   - 资金态度 = 大盘主力（整体）+ 行业资金（结构性）")
print("   - 风险信号 = 炸板率")
print()

# ========== 情绪评分：基于陈小群战法核心逻辑 ==========
# 权重分配（基于第一性原理）：
# - 涨停家数：40%（赚钱效应，最重要）
# - 连板高度：20%（情绪强度验证）
# - 炸板率：20%（风险信号）
# - 大盘主力：10%（整体资金态度）
# - 行业资金：10%（结构性机会）

sentiment_score = 0
sentiment_factors = []

# 1. 大盘主力净流入（整体资金态度，权重10%）
# 调整逻辑：大盘流出应该更严重，分级更细
if '大盘' in results:
    pct = results['大盘']['主力净流入(%)']
    if pct > 2:
        sentiment_score += 1.0
        sentiment_factors.append(f"大盘主力大幅净流入 {pct:.1f}% (整体资金非常积极)")
    elif pct > 1:
        sentiment_score += 0.8
        sentiment_factors.append(f"大盘主力净流入 {pct:.1f}% (整体资金积极)")
    elif pct > 0:
        sentiment_score += 0.5
        sentiment_factors.append(f"大盘主力小幅流入 {pct:.1f}% (整体资金略积极)")
    elif pct > -1:
        sentiment_score += 0.0
        sentiment_factors.append(f"大盘主力小幅流出 {pct:.1f}% (整体资金略谨慎)")
    elif pct > -3:
        sentiment_score -= 1.0
        sentiment_factors.append(f"大盘主力流出 {pct:.1f}% (整体资金谨慎)")
    else:
        sentiment_score -= 1.5  # 大幅流出，扣更多分
        sentiment_factors.append(f"大盘主力大幅流出 {pct:.1f}% (整体资金非常谨慎)")

# 2. 行业资金净流入（结构性机会，权重10%）
# 调整逻辑：行业资金流入不能完全抵消大盘流出，只能部分缓解
if '行业' in results:
    sec_in = results['行业']['总净流入(亿)']
    if sec_in > 200:
        sentiment_score += 0.8  # 降低权重，不能完全抵消大盘流出
        sentiment_factors.append(f"行业主力净流入较强 {sec_in:.0f}亿（结构性偏多，但不足以抵消大盘流出）")
    elif sec_in > 100:
        sentiment_score += 0.5
        sentiment_factors.append(f"行业主力净流入 {sec_in:.0f}亿（结构性略多）")
    elif sec_in < -200:
        sentiment_score -= 1.0
        sentiment_factors.append(f"行业主力净流出较强 {abs(sec_in):.0f}亿（结构性偏空）")
    elif sec_in < -100:
        sentiment_score -= 0.5
        sentiment_factors.append(f"行业主力净流出 {abs(sec_in):.0f}亿（结构性略空）")
    else:
        sentiment_score += 0.0
        sentiment_factors.append(f"行业资金流向中性 {sec_in:.0f}亿")

# 判断结果（资金态度部分）
# 调整阈值：优先识别结构性机会（大盘流出但行业资金流入）
# 先检查是否有结构性机会（大盘流出但行业资金流入>100亿）
has_structural_opportunity = False
if '大盘' in results and '行业' in results:
    market_pct = results['大盘']['主力净流入(%)']
    sector_in = results['行业']['总净流入(亿)']
    # 结构性机会：大盘流出但行业资金流入较多
    if market_pct < -1 and sector_in > 100:
        has_structural_opportunity = True

if has_structural_opportunity:
    fund_sentiment, fund_emoji = "结构性机会", "🔄"  # 结构性做多 + 指数承压
elif sentiment_score >= 1.0:
    fund_sentiment, fund_emoji = "积极", "📈"
elif sentiment_score >= 0.3:
    fund_sentiment, fund_emoji = "略积极", "📈"
elif sentiment_score <= -1.0:
    fund_sentiment, fund_emoji = "谨慎", "📉"
elif sentiment_score <= -0.3:
    fund_sentiment, fund_emoji = "略谨慎", "📉"
else:
    fund_sentiment, fund_emoji = "中性", "➡️"

print(f"📊 资金态度评分: {sentiment_score:.1f} (大盘主力 + 行业资金)")
print(f"{fund_emoji} 资金态度: {fund_sentiment}")

# 如果是结构性机会，给出更详细的说明
if fund_sentiment == "结构性机会":
    print("\n💡 结构性机会解读：")
    if '大盘' in results and '行业' in results:
        market_pct = results['大盘']['主力净流入(%)']
        sector_in = results['行业']['总净流入(亿)']
        print(f"   • 指数层面：大盘主力流出 {abs(market_pct):.1f}%，整体承压")
        print(f"   • 结构层面：行业资金净流入 {sector_in:.0f}亿，有结构性机会")
        print(f"   • 策略建议：可关注资金流入的行业（如：{', '.join(results['行业'].get('前3行业', []))}）")
        print(f"   • 风险提示：整体市场承压，需控制仓位，避免追高")

print("\n资金态度判断因素:")
for factor in sentiment_factors:
    print(f"  • {factor}")

# 保存结果供后续使用（用于judge_emotion_cycle函数）
avg_inflow = results.get('大盘', {}).get('主力净流入(%)', 0)
# 北向资金暂时不使用
north_flow = 0

# ========== 3. JQData资金流向估算（可选，免费方案） ==========
# 注意：JQData的get_money_flow_pro接口需要付费权限，暂时不可用
# 这里提供一个基于价格和成交量的免费估算方案（可选）

print("\n" + "=" * 80)
print("📊 3. JQData资金流向估算（免费方案，可选）")
print("=" * 80)
print("\n💡 说明：")
print("   • JQData的get_money_flow_pro接口需要付费权限，暂时不可用")
print("   • 使用价格和成交量估算资金流向（免费方案）")
print("   • 公式: main_flow = (price_position - 0.5) * money")
print("   • 其中: price_position = (close - low) / (high - low)")
print("   • 仅供参考，精度不如专业资金流向接口")

try:
    if jq is None:
        print("\n⚠️  JQData未连接，跳过估算")
    else:
        # 使用价格和成交量估算大盘资金流向
        cn_today = cn_today_str()
        
        # 获取沪深300指数数据
        indices = ['000300.XSHG']  # 沪深300
        
        total_estimated_flow = 0.0
        valid_count = 0
        
        for index_code in indices:
            try:
                # 获取最近5个交易日的数据
                prices = jq.get_price(
                    index_code,
                    end_date=cn_today,
                    count=5,
                    fields=['close', 'high', 'low', 'volume', 'money']
                )
                
                if prices is not None and not prices.empty:
                    # 计算价格位置
                    prices['price_position'] = (prices['close'] - prices['low']) / (prices['high'] - prices['low'] + 1e-8)
                    # 估算资金流向（价格位置高表示资金流入）
                    prices['estimated_flow'] = (prices['price_position'] - 0.5) * prices['money']
                    
                    # 取最新一日的估算值
                    latest_flow = prices['estimated_flow'].iloc[-1] / 1e8  # 转换为亿元
                    total_estimated_flow += latest_flow
                    valid_count += 1
                    
            except Exception as e:
                print(f"  ⚠️  获取{index_code}数据失败: {str(e)[:80]}")
                continue
        
        if valid_count > 0:
            print(f"\n✅ JQData资金流向估算完成")
            print(f"   日期: {cn_today}")
            print(f"   估算主力净流入: {total_estimated_flow:.2f}亿元")
            print(f"   算法: 基于价格位置和成交额估算")
            print(f"   说明: 此方法精度较低，仅供参考")
            
            # 与AKShare对比（如果可用）
            if '大盘' in results:
                ak_market = results['大盘']
                ak_inflow = ak_market.get('主力净流入(亿)', 0)
                
                print(f"\n📊 数据对比（JQData估算 vs AKShare）:")
                print(f"   JQData估算: {total_estimated_flow:.2f}亿元")
                print(f"   AKShare: {ak_inflow:.2f}亿元")
                diff = total_estimated_flow - ak_inflow
                if abs(ak_inflow) > 0.01:
                    diff_pct = (diff / abs(ak_inflow)) * 100
                    print(f"   差异: {diff:.2f}亿元 ({diff_pct:.1f}%)")
                    print(f"   ⚠️  估算方法精度较低，差异可能较大")
                    print(f"   💡 建议：以AKShare数据为准（更准确）")
                else:
                    print(f"   ⚠️  AKShare数据为0或接近0，无法对比")
        else:
            print(f"\n⚠️  无法获取数据，跳过估算")
            
except Exception as e:
    print(f"\n⚠️  JQData资金流向估算失败: {str(e)[:100]}")
    print(f"   建议: 使用AKShare数据作为主要数据源")

print("\n" + "=" * 80)
print("✅ 整合测试完成！")
print("=" * 80)


🧪 资金流向数据获取整合测试（完整版）
当前时间: 2026-01-14 07:47:02

📊 1. 大盘资金流向
📅 数据范围: 2025-07-18 至 2026-01-14 (共121个交易日)
✅ 找到目标日期数据: 2026-01-14

📊 大盘资金流向数据:
   日期: 2026-01-14
   上证涨跌幅: -0.31%
   深证涨跌幅: 0.56%
   主力净流入: -889.40亿元 (-2.26%)
   超大单净流入: -352.79亿元 (-0.90%)
   大单净流入: -536.61亿元 (-1.36%)
   中单净流入: 18.06亿元 (0.05%)
   小单净流入: 871.34亿元 (2.21%)

📊 2. 行业资金流向（结构性机会指标）
💡 说明: '今日'指最新交易日，如果目标日期是最新交易日，可以获取
   目标日期: 2026-01-14

  2.1 尝试'今日'数据（最新交易日）


  0%|          | 0/1 [00:00<?, ?it/s]

  ✅ '今日'数据有效，行业数: 86
     行业总主力净流入: -878.38亿元
     主力流入前3: 互联网服务, 软件开发, 化学原料

     前3名行业详情:
       互联网服务: 67.59亿元 (2.44%)
       软件开发: 42.01亿元 (1.59%)
       化学原料: 6.79亿元 (2.23%)

📈 数据获取汇总

大盘:
  日期: 2026-01-14
  主力净流入(亿): -889.40
  主力净流入(%): -2.26
  超大单净流入(亿): -352.79
  大单净流入(亿): -536.61
  数据源: AKShare stock_market_fund_flow

行业:
  总净流入(亿): -878.38
  行业数: 86
  前3行业: 互联网服务, 软件开发, 化学原料
  数据源: 今日（最新交易日）
  日期: 2026-01-14

💡 数据源说明:
   大盘资金流向: AKShare stock_market_fund_flow
   行业资金流向: 今日（最新交易日）

🎯 综合情绪判断（基于第一性原理）

💡 第一性原理分析：
   市场情绪 = 赚钱效应 + 资金态度 + 风险信号
   - 赚钱效应 = 涨停家数（最重要）+ 连板高度（验证）
   - 资金态度 = 大盘主力（整体）+ 行业资金（结构性）
   - 风险信号 = 炸板率

📊 资金态度评分: -2.0 (大盘主力 + 行业资金)
📉 资金态度: 谨慎

资金态度判断因素:
  • 大盘主力流出 -2.3% (整体资金谨慎)
  • 行业主力净流出较强 878亿（结构性偏空）

📊 3. JQData资金流向估算（免费方案，可选）

💡 说明：
   • JQData的get_money_flow_pro接口需要付费权限，暂时不可用
   • 使用价格和成交量估算资金流向（免费方案）
   • 公式: main_flow = (price_position - 0.5) * money
   • 其中: price_position = (close - low) / (high - low)
   • 仅供参考，精度不如专业资金流向接口
  ⚠️  获取000300.XSHG数据失败: 

## 🎯 5. 判断情绪周期

In [40]:
def judge_emotion_cycle(limit_up_count, max_height, zhaban_rate, avg_inflow, fund_sentiment_score=0):
    """
    判断市场情绪周期 - 基于第一性原理 + 陈小群战法
    
    第一性原理分析：
    市场情绪 = 赚钱效应 + 资金态度 + 风险信号
    
    核心指标权重分配（基于第一性原理）：
    1. 涨停家数：40%（赚钱效应，最重要）
    2. 连板高度：20%（情绪强度验证）
    3. 炸板率：20%（风险信号）
    4. 资金态度：20%（大盘主力 + 行业资金，已在Cell 13计算）
    
    判断标准（根据知识库）:
    - 退潮期: <10只, <3板, >40%炸板率
    - 启动期: 10-30只, 3-4板, 10-20%炸板率
    - 加速期: 30-60只, 4-6板, 15-25%炸板率
    - 过热期: >60只, >7板, >30%炸板率
    """
    
    # 初始化置信度分数（总分5.0）
    confidence_score = 0
    factors = []
    
    # ========== 1. 主要依据：涨停家数（权重40%，最重要） ==========
    if limit_up_count < 10:
        cycle = "退潮期"
        position = "0%"
        strategy = "空仓等待"
        confidence_score += 2.0  # 40%权重，满分2.0
        factors.append(f"涨停家数{limit_up_count}只（<10只，退潮期特征，权重40%）")
    elif limit_up_count < 30:
        cycle = "启动期"
        position = "10%"
        strategy = "首板卡位术（10%试错仓）"
        confidence_score += 2.0
        factors.append(f"涨停家数{limit_up_count}只（10-30只，启动期特征，权重40%）")
    elif limit_up_count < 60:
        cycle = "加速期"
        position = "50%+"
        strategy = "龙头战法（重仓持有）"
        confidence_score += 2.0
        factors.append(f"涨停家数{limit_up_count}只（30-60只，加速期特征，权重40%）")
    else:
        cycle = "过热期"
        position = "30-50%"
        strategy = "逐步减仓"
        confidence_score += 2.0
        factors.append(f"涨停家数{limit_up_count}只（>60只，过热期特征，权重40%）")
    
    # ========== 2. 连板高度验证（权重20%，情绪强度验证） ==========
    height_score = 0
    if cycle == "退潮期" and max_height < 3:
        height_score = 1.0  # 20%权重，满分1.0
        factors.append(f"连板高度{max_height}板（<3板，符合退潮期，权重20%）")
    elif cycle == "启动期" and 3 <= max_height <= 4:
        height_score = 1.0
        factors.append(f"连板高度{max_height}板（3-4板，符合启动期，权重20%）")
    elif cycle == "加速期" and 4 <= max_height <= 6:
        height_score = 1.0
        factors.append(f"连板高度{max_height}板（4-6板，符合加速期，权重20%）")
    elif cycle == "过热期" and max_height > 7:
        height_score = 1.0
        factors.append(f"连板高度{max_height}板（>7板，符合过热期，权重20%）")
    elif cycle == "启动期" and max_height >= 3 and limit_up_count >= 25:
        # 有3板以上且涨停数接近30，可能进入加速期
        cycle = "加速期"
        position = "50%+"
        strategy = "龙头战法（重仓持有）"
        height_score = 1.0
        factors.append(f"连板高度{max_height}板+涨停数{limit_up_count}只（接近加速期，权重20%）")
    elif max_height == 0:
        height_score = 0.3  # 无连板数据，给部分分
        factors.append(f"连板高度{max_height}板（无连板数据，权重20%，部分得分）")
    else:
        # 连板高度与周期不完全匹配，给部分分
        height_score = 0.5
        factors.append(f"连板高度{max_height}板（与周期不完全匹配，权重20%，部分得分）")
    
    confidence_score += height_score
    
    # ========== 3. 炸板率验证（权重20%，风险信号） ==========
    risk_score = 0
    if cycle == "退潮期" and zhaban_rate > 40:
        risk_score = 1.0  # 20%权重，满分1.0
        factors.append(f"炸板率{zhaban_rate:.1f}%（>40%，确认退潮期，权重20%）")
    elif cycle == "启动期" and 10 <= zhaban_rate <= 20:
        risk_score = 1.0
        factors.append(f"炸板率{zhaban_rate:.1f}%（10-20%，符合启动期，权重20%）")
    elif cycle == "加速期" and 15 <= zhaban_rate <= 25:
        risk_score = 1.0
        factors.append(f"炸板率{zhaban_rate:.1f}%（15-25%，符合加速期，权重20%）")
    elif cycle == "过热期" and zhaban_rate > 30:
        risk_score = 1.0
        factors.append(f"炸板率{zhaban_rate:.1f}%（>30%，符合过热期，权重20%）")
    elif cycle == "加速期" and zhaban_rate > 30:
        # 炸板率过高，可能进入过热期
        cycle = "过热期"
        position = "30-50%"
        strategy = "逐步减仓"
        risk_score = 1.0
        factors.append(f"炸板率{zhaban_rate:.1f}%（>30%，可能过热，权重20%）")
    elif cycle == "退潮期" and zhaban_rate <= 40:
        risk_score = 0.5  # 退潮期但炸板率不高，给部分分
        factors.append(f"炸板率{zhaban_rate:.1f}%（退潮期但炸板率不高，权重20%，部分得分）")
    else:
        # 炸板率与周期不完全匹配，给部分分
        risk_score = 0.5
        factors.append(f"炸板率{zhaban_rate:.1f}%（与周期不完全匹配，权重20%，部分得分）")
    
    confidence_score += risk_score
    
    # ========== 4. 资金态度验证（权重20%，大盘主力+行业资金） ==========
    # fund_sentiment_score已在Cell 13计算（范围-2.0到+2.0），归一化到0-1.0
    if fund_sentiment_score != 0:
        # 将-2.0~+2.0映射到0~1.0
        fund_score = (fund_sentiment_score + 2.0) / 4.0  # 归一化到0-1.0
        confidence_score += fund_score
        if fund_sentiment_score > 0:
            factors.append(f"资金态度积极（评分{fund_sentiment_score:.1f}，权重20%）")
        else:
            factors.append(f"资金态度谨慎（评分{fund_sentiment_score:.1f}，权重20%）")
    else:
        # 资金态度中性，给部分分
        fund_score = 0.5
        confidence_score += fund_score
        factors.append(f"资金态度中性（权重20%，部分得分）")
    
    # 传统资金流向验证（保留作为补充，但不计入主要评分）
    if avg_inflow > 0.5 and cycle in ["启动期", "加速期"]:
        factors.append(f"💡 补充：大盘主力净流入{avg_inflow:.2f}%（支持上涨周期）")
    elif avg_inflow < -0.5 and cycle == "退潮期":
        factors.append(f"💡 补充：大盘主力净流出{avg_inflow:.2f}%（确认退潮期）")
    elif avg_inflow < -0.5 and cycle != "退潮期":
        factors.append(f"⚠️  补充：大盘主力净流出{avg_inflow:.2f}%（与周期判断不一致，需注意）")
    
    # 计算置信度等级
    if confidence_score >= 4:
        confidence_level = "高"
        confidence_icon = "🟢"
    elif confidence_score >= 3:
        confidence_level = "中"
        confidence_icon = "🟡"
    else:
        confidence_level = "低"
        confidence_icon = "🔴"
    
    return {
        'cycle': cycle,
        'position': position,
        'strategy': strategy,
        'limit_up_count': limit_up_count,
        'max_height': max_height,
        'zhaban_rate': zhaban_rate,
        'avg_inflow': avg_inflow,
        'confidence_score': confidence_score,
        'confidence_level': confidence_level,
        'confidence_icon': confidence_icon,
        'factors': factors
    }

# 判断当前市场情绪周期
# 获取资金态度评分（从Cell 13的结果中提取）
fund_sentiment_score = sentiment_score  # 从Cell 13获取的资金态度评分
result = judge_emotion_cycle(limit_up_count, max_height, zhaban_rate, avg_inflow, fund_sentiment_score)

print("=" * 70)
print("🎯 市场情绪周期判断结果（基于第一性原理 + 陈小群战法）")
print("=" * 70)
print()
print("💡 第一性原理分析框架：")
print("   市场情绪 = 赚钱效应 + 资金态度 + 风险信号")
print("   - 赚钱效应：涨停家数（40%）+ 连板高度（20%）")
print("   - 风险信号：炸板率（20%）")
print("   - 资金态度：大盘主力 + 行业资金（20%）")
print()
print(f"📊 市场指标:")
print(f"   涨停家数: {result['limit_up_count']}只（赚钱效应，权重40%）")
print(f"   最高连板: {result['max_height']}板（情绪强度验证，权重20%）")
print(f"   炸板率: {result['zhaban_rate']:.2f}%（风险信号，权重20%）")
print(f"   资金态度评分: {fund_sentiment_score:.1f}（大盘+行业，权重20%）")
print(f"   大盘主力净流入: {result['avg_inflow']:.2f}%（补充参考）")
print()
print(f"🎯 判断结果:")
print(f"   情绪周期: {result['cycle']}")
print(f"   建议仓位: {result['position']}")
print(f"   推荐策略: {result['strategy']}")
print()
print(f"📈 判断依据（按权重排序）:")
for factor in result['factors']:
    print(f"   • {factor}")
print()
print(f"🎯 置信度: {result['confidence_icon']} {result['confidence_level']} ({result['confidence_score']:.2f}/5.00)")
print()
print("=" * 70)

🎯 市场情绪周期判断结果（基于第一性原理 + 陈小群战法）

💡 第一性原理分析框架：
   市场情绪 = 赚钱效应 + 资金态度 + 风险信号
   - 赚钱效应：涨停家数（40%）+ 连板高度（20%）
   - 风险信号：炸板率（20%）
   - 资金态度：大盘主力 + 行业资金（20%）

📊 市场指标:
   涨停家数: 102只（赚钱效应，权重40%）
   最高连板: 5板（情绪强度验证，权重20%）
   炸板率: 36.65%（风险信号，权重20%）
   资金态度评分: -2.0（大盘+行业，权重20%）
   大盘主力净流入: -2.26%（补充参考）

🎯 判断结果:
   情绪周期: 过热期
   建议仓位: 30-50%
   推荐策略: 逐步减仓

📈 判断依据（按权重排序）:
   • 涨停家数102只（>60只，过热期特征，权重40%）
   • 连板高度5板（与周期不完全匹配，权重20%，部分得分）
   • 炸板率36.6%（>30%，符合过热期，权重20%）
   • 资金态度谨慎（评分-2.0，权重20%）
   • ⚠️  补充：大盘主力净流出-2.26%（与周期判断不一致，需注意）

🎯 置信度: 🟡 中 (3.50/5.00)



## 📈 6. 可视化分析

In [41]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 检查必需的变量是否已定义
def var_exists(var_name):
    """检查变量是否存在于全局或局部作用域"""
    return var_name in globals() or var_name in locals()

def get_var_safe(var_name):
    """安全获取变量，如果不存在则返回None"""
    if var_name in globals():
        return globals()[var_name]
    elif var_name in locals():
        return locals()[var_name]
    else:
        return None

# 检查哪些变量缺失
required_vars = {
    'limit_up_count': '涨停家数',
    'result': '情绪周期判断结果',
    'zhaban_rate': '炸板率',
    'limit_up_data': '涨停板数据',
    'today_display': '当前日期'
}

missing_vars = []
for var_name, var_desc in required_vars.items():
    if not var_exists(var_name):
        missing_vars.append(f"{var_name} ({var_desc})")

if missing_vars:
    print("=" * 80)
    print("⚠️  缺少必需的变量，请先运行前面的cell:")
    print("=" * 80)
    for var in missing_vars:
        print(f"   ❌ {var}")
    print("\n💡 提示: 请按顺序运行以下cell:")
    print("   1. Cell 7: 获取涨停板数据")
    print("   2. Cell 9: 计算连板高度")
    print("   3. Cell 11: 计算炸板率")
    print("   4. Cell 13: 判断情绪周期")
    print("   5. Cell 19: 绘制图表（当前cell）")
    print("\n" + "=" * 80)
    raise NameError(f"缺少必需的变量: {', '.join([v.split(' ')[0] for v in missing_vars])}")

# 获取变量值（此时变量已存在）
limit_up_count = get_var_safe('limit_up_count')
result = get_var_safe('result')
zhaban_rate = get_var_safe('zhaban_rate')
limit_up_data = get_var_safe('limit_up_data')
today_display = get_var_safe('today_display')

# 设置默认值（容错处理，防止值为None）
if limit_up_count is None:
    limit_up_count = 0
if zhaban_rate is None:
    zhaban_rate = 0
if result is None:
    result = {'cycle': '未知', 'confidence_icon': '⚪', 'confidence_level': '未知'}
if today_display is None:
    from datetime import datetime
    today_display = datetime.now().strftime('%Y-%m-%d')

print("✅ 所有必需变量已检查，准备绘制图表...")

# 创建可视化图表
# 优化布局：增加间距，缩短标题，防止文字重叠
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('涨停家数', '连板高度', '炸板率', '情绪周期'),  # 缩短标题
    specs=[[{"type": "indicator"}, {"type": "xy"}],  # row1 col2改为xy类型
           [{"type": "indicator"}, {"type": "domain"}]],  # row2 col2改为domain类型（用于饼图）
    vertical_spacing=0.22,  # 垂直间距
    horizontal_spacing=0.18,  # 水平间距
    row_heights=[0.52, 0.48],  # 给第2行更多高度，避免Pie顶部靠近分隔线
    column_widths=[0.5, 0.5]  # 明确列宽比例
)

# 1. 涨停家数（仪表盘）
cycle_ranges = {
    '退潮期': (0, 10, 'lightgray'),
    '启动期': (10, 30, 'lightyellow'),
    '加速期': (30, 60, 'lightgreen'),
    '过热期': (60, 100, 'lightcoral')
}

fig.add_trace(
    go.Indicator(
        mode="gauge+number+delta",
        value=limit_up_count,
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': f"涨停家数<br><span style='font-size:0.8em'>{result.get('cycle', '未知')}</span>"},
        gauge={
            'axis': {'range': [None, 100]},
            'bar': {'color': "darkblue"},
            'steps': [
                {'range': [0, 10], 'color': "lightgray", 'name': '退潮期'},
                {'range': [10, 30], 'color': "lightyellow", 'name': '启动期'},
                {'range': [30, 60], 'color': "lightgreen", 'name': '加速期'},
                {'range': [60, 100], 'color': "lightcoral", 'name': '过热期'}
            ],
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': limit_up_count
            }
        },
        delta={'reference': 30, 'position': "top"}
    ),
    row=1, col=1
)

# 2. 连板高度分布（固定在row=1, col=2）
if limit_up_data is not None and not limit_up_data.empty and '连板高度' in limit_up_data.columns:
    height_data = limit_up_data[limit_up_data['连板高度'] > 0]
    if not height_data.empty:
        height_dist = height_data['连板高度'].value_counts().sort_index()
        fig.add_trace(
            go.Bar(
                x=[f"{int(i)}板" for i in height_dist.index],
                y=height_dist.values,
                name="连板高度分布",
                marker_color='lightblue',
                text=height_dist.values,
                textposition='outside',
                showlegend=False  # 禁用legend，避免文本出现在右下角
            ),
            row=1, col=2
        )
    else:
        # 如果没有连板数据，显示空图表提示
        fig.add_trace(
            go.Bar(x=[], y=[], showlegend=False),
            row=1, col=2
        )
else:
    fig.add_trace(
        go.Bar(x=[], y=[], showlegend=False),
        row=1, col=2
    )

# 3. 炸板率（仪表盘）
zhaban_ranges = [
    {'range': [0, 20], 'color': "lightgreen", 'name': '正常'},
    {'range': [20, 40], 'color': "lightyellow", 'name': '偏高'},
    {'range': [40, 100], 'color': "lightcoral", 'name': '过高'}
]

fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=zhaban_rate,
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': "炸板率 (%)"},
        gauge={
            'axis': {'range': [None, 100]},
            'bar': {'color': "darkgreen"},
            'steps': zhaban_ranges,
            'threshold': {
                'line': {'color': "red", 'width': 3},
                'thickness': 0.75,
                'value': 40  # 40%作为风险阈值
            }
        }
    ),
    row=2, col=1
)

# 4. 情绪周期（环形图）- 四状态都显示，高亮当前状态
cycle_labels = ['退潮期', '启动期', '加速期', '过热期']
cycle_colors = ['lightgray', 'lightyellow', 'lightgreen', 'lightcoral']
cycle_values = [1, 1, 1, 1]  # 等分，只做状态指示，不表达占比

# 安全获取当前周期
current_cycle = result.get('cycle', '未知')
if current_cycle not in cycle_labels:
    # 如果周期不在列表中，默认显示第一个
    print(f"⚠️  未知的情绪周期: {current_cycle}，使用默认值")
    current_cycle = cycle_labels[0]

# 计算pull值：当前周期突出显示，其他不突出
cycle_idx = cycle_labels.index(current_cycle)
pull_values = [0.08 if i == cycle_idx else 0 for i in range(len(cycle_labels))]

# 调整颜色：当前周期保持原色，其他降低透明度
marker_colors_adjusted = []
for i, color in enumerate(cycle_colors):
    if i == cycle_idx:
        marker_colors_adjusted.append(color)  # 当前周期保持原色
    else:
        # 其他扇区降低透明度（使用rgba格式，根据原色调整）
        if color == 'lightgray':
            marker_colors_adjusted.append("rgba(211,211,211,0.3)")  # lightgray半透明
        elif color == 'lightyellow':
            marker_colors_adjusted.append("rgba(255,255,224,0.3)")  # lightyellow半透明
        elif color == 'lightgreen':
            marker_colors_adjusted.append("rgba(144,238,144,0.3)")  # lightgreen半透明
        elif color == 'lightcoral':
            marker_colors_adjusted.append("rgba(240,128,128,0.3)")  # lightcoral半透明
        else:
            marker_colors_adjusted.append("rgba(200,200,200,0.3)")  # 默认浅灰色半透明

# 创建环形图：固定显示4个扇区，等分
fig.add_trace(
    go.Pie(
        labels=cycle_labels,
        values=cycle_values,
        marker_colors=marker_colors_adjusted,  # 使用调整后的颜色（当前周期原色，其他半透明）
        hole=0.55,  # 环形图中心孔
        textinfo='none',  # 扇区文字不显示，避免重叠
        hovertemplate='<b>%{label}</b><extra></extra>',  # 只显示label，不显示percent（因为等分会误导）
        pull=pull_values,  # 当前周期突出显示（pull=0.08）
        showlegend=False  # 不在饼图显示图例
    ),
    row=2, col=2
)

# 在饼图中心添加注释，显示当前周期和置信度
fig.add_annotation(
    x=0.75, y=0.20, xref="paper", yref="paper",  # 对应row=2, col=2的中心位置
    text=f"<b>当前周期：{current_cycle}</b><br><span style='font-size:0.85em'>置信度：{result.get('confidence_icon', '⚪')} {result.get('confidence_level', '未知')}</span>",
    showarrow=False,
    align="center",
    font=dict(size=13, color="black"),
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="gray",
    borderwidth=1,
    borderpad=4
)

# 为所有周期添加文本标注，并在当前周期位置添加箭头
# 计算每个扇区的位置（环形图，4等分，每个扇区90度）
# 扇区中心角度：0度（退潮期，右侧），90度（启动期，上方），180度（加速期，左侧），270度（过热期，下方）
# 标注位置：在环形图外侧，箭头从环形图边缘指向标注文本
cycle_positions = {
    0: {'text_x': 0.88, 'text_y': 0.20, 'arrow_offset_x': -40, 'arrow_offset_y': 0},  # 退潮期：右侧，箭头向左
    1: {'text_x': 0.75, 'text_y': 0.32, 'arrow_offset_x': 0, 'arrow_offset_y': -30},  # 启动期：上方，箭头向下
    2: {'text_x': 0.62, 'text_y': 0.20, 'arrow_offset_x': 40, 'arrow_offset_y': 0},   # 加速期：左侧，箭头向右
    3: {'text_x': 0.75, 'text_y': 0.08, 'arrow_offset_x': 0, 'arrow_offset_y': 30}    # 过热期：下方，箭头向上
}

for i, label in enumerate(cycle_labels):
    pos = cycle_positions[i]
    is_current = (i == cycle_idx)
    
    # 添加周期名称标注
    fig.add_annotation(
        x=pos['text_x'], y=pos['text_y'], xref="paper", yref="paper",
        text=f"<b>{label}</b>",
        showarrow=False,
        align="center",
        font=dict(size=11, color="black" if is_current else "gray"),
        bgcolor="rgba(255,255,255,0.9)" if is_current else "rgba(255,255,255,0.6)",
        bordercolor="red" if is_current else "gray",
        borderwidth=2 if is_current else 1,
        borderpad=4
    )
    
    # 如果是当前周期，添加箭头指向（从环形图边缘指向标注文本）
    if is_current:
        # 使用像素偏移量（ax和ay使用像素单位，axref和ayref默认是'pixel'）
        fig.add_annotation(
            x=pos['text_x'], y=pos['text_y'], xref="paper", yref="paper",  # 箭头终点（标注文本位置）
            ax=pos['arrow_offset_x'], ay=pos['arrow_offset_y'],  # 箭头起点（使用像素偏移量，从终点向起点方向）
            text="",  # 箭头不需要文本
            showarrow=True,
            arrowhead=2,
            arrowsize=1.8,
            arrowwidth=2.5,
            arrowcolor="red",
            standoff=3  # 箭头与标注的距离
        )

# 更新布局
fig.update_layout(
    title_text=f"陈小群战法 - 市场情绪周期分析<br><span style='font-size:0.7em'>{today_display} | 置信度: {result.get('confidence_icon', '⚪')} {result.get('confidence_level', '未知')}</span>",
    height=800,  # 增加高度（从900增加到1000），提供更多空间
    showlegend=False,  # 禁用全局图例
    font=dict(size=11),  # 稍微减小字体（从12到11），防止重叠
    margin=dict(l=50, r=50, t=100, b=50),  # 边距设置
    title=dict(
        x=0.5,  # 标题居中
        xanchor='center',
        font=dict(size=14)  # 标题字体大小
    )
)

# 优化subplot标题字体大小，防止重叠
fig.update_annotations(
    font=dict(size=10),  # subplot标题字体大小（略小，避免压到图形）
    yshift=22  # 增加上移距离，增加与图表内容的间距
)

# 更新坐标轴标签（连板高度分布固定在row=1, col=2）
fig.update_xaxes(title_text="连板高度", row=1, col=2)
fig.update_yaxes(title_text="股票数量", row=1, col=2)

# 显示图表
# 在Jupyter Notebook中，plotly会自动检测并使用合适的渲染器
fig.show()

✅ 所有必需变量已检查，准备绘制图表...


## 💡 7. 策略建议

In [42]:
print("=" * 70)
print(f"💡 基于情绪周期的策略建议")
print("=" * 70)
print()

cycle = result['cycle']

if cycle == "退潮期":
    print("📋 当前处于退潮期，建议：")
    print("   1. ✅ 空仓等待，不要操作")
    print("   2. ✅ 每日监控涨停家数，等待情绪周期转换")
    print("   3. ✅ 关注市场情绪指标的改善")
    print("   4. ⚠️  退潮期操作风险极高，即使符合条件成功率也会大幅降低")
    print()
    print("📝 后续步骤：")
    print("   • 等待涨停家数>10只再考虑操作")
    print("   • 等待连板高度>=3板")
    print("   • 等待炸板率<30%")

elif cycle == "启动期":
    print("📋 当前处于启动期，建议：")
    print("   1. ✅ 使用首板卡位术（10%试错仓）")
    print("   2. ✅ 筛选早盘9:35前涨停的股票")
    print("   3. ✅ 检查选股条件：")
    print("      • 流通市值<30亿")
    print("      • 封单量>流通市值2%")
    print("      • 题材新颖、有想象空间")
    print("      • 板块内至少3只跟风涨停")
    print("   4. ✅ 严格止损：次日不涨停立即止损（-5%）")
    print()
    print("📝 后续步骤：")
    print("   • 每日早盘9:35前扫描涨停股票")
    print("   • 确认板块效应后再介入")
    print("   • 如果次日二板，考虑使用二板定龙术（50%）")

elif cycle == "加速期":
    print("📋 当前处于加速期，建议：")
    print("   1. ✅ 使用龙头战法（50%+重仓）")
    print("   2. ✅ 识别市场总龙头")
    print("   3. ✅ 在龙头启动期重仓介入")
    print("   4. ✅ 坚定持有，不爱做T，看准就持有到巅峰")
    print("   5. ⚠️  总仓位建议不超过70%")
    print()
    print("📝 后续步骤：")
    print("   • 寻找板块内涨幅最大或最早涨停的股票")
    print("   • 确认龙头地位后再重仓介入")
    print("   • 持有至见顶或出现风险信号")

elif cycle == "过热期":
    print("📋 当前处于过热期，建议：")
    print("   1. ✅ 逐步减仓（保留30-50%仓位）")
    print("   2. ✅ 监控炸板率，如果>30%大幅减仓")
    print("   3. ✅ 关注连板高度的变化")
    print("   4. ✅ 如果板块效应减弱，立即止盈")
    print("   5. ⚠️  过热期风险极高，随时可能见顶")
    print()
    print("📝 后续步骤：")
    print("   • 准备空仓等待退潮期结束")
    print("   • 不要贪心，及时止盈保住收益")
    print("   • 关注市场情绪指标的恶化")

print()
print("=" * 70)

💡 基于情绪周期的策略建议

📋 当前处于过热期，建议：
   1. ✅ 逐步减仓（保留30-50%仓位）
   2. ✅ 监控炸板率，如果>30%大幅减仓
   3. ✅ 关注连板高度的变化
   4. ✅ 如果板块效应减弱，立即止盈
   5. ⚠️  过热期风险极高，随时可能见顶

📝 后续步骤：
   • 准备空仓等待退潮期结束
   • 不要贪心，及时止盈保住收益
   • 关注市场情绪指标的恶化



## 📊 8. 算法设计与验证

### 8.1 算法设计要点

本Notebook实现了陈小群战法的市场环境判断算法，核心特点：

1. **多维度综合判断**
   - 涨停家数（主要依据，权重最高）
   - 连板高度（验证指标，权重中等）
   - 炸板率（风险指标，权重中等）
   - 资金流向（辅助指标，权重较低）

2. **置信度评分系统**
   - 根据各项指标的符合程度计算置信度
   - 高置信度（≥4.0）：判断可靠，可执行策略
   - 中置信度（3.0-4.0）：判断基本可靠，需要谨慎
   - 低置信度（<3.0）：判断不确定，建议观察

3. **动态调整机制**
   - 当涨停家数处于边界值（如25只），结合连板高度动态调整
   - 当炸板率异常时，自动调整周期判断

### 8.2 数据验证方法

所有数据都可以通过以下方法验证：

1. **涨停板数据**：`ak.stock_zt_pool_em(date=today)`
   - 来源：AKShare实时数据
   - 验证：检查数据完整性和有效性

2. **连板高度**：`jq.get_price(code, count=5, frequency='daily')`
   - 来源：JQData历史价格数据
   - 验证：计算连续涨停天数

3. **炸板率**：统计涨跌幅>=9%但<10%的股票
   - 来源：AKShare实时行情
   - 验证：计算炸板股票占比

4. **资金流向**：`jq.get_money_flow(index, start_date, end_date)`
   - 来源：JQData资金流向数据
   - 验证：计算资金净流入百分比

### 8.3 策略展示

根据判断的情绪周期，本Notebook会：

1. **显示详细判断结果**
   - 市场指标（涨停家数、连板高度、炸板率、资金流向）
   - 判断结果（情绪周期、建议仓位、推荐策略）
   - 判断依据（各项指标的符合情况）
   - 置信度评估（高/中/低）

2. **生成可视化图表**
   - 涨停家数仪表盘（显示当前周期）
   - 连板高度分布柱状图
   - 炸板率仪表盘（显示风险水平）
   - 情绪周期饼图（显示当前状态）

3. **提供策略建议**
   - 基于当前情绪周期的具体操作建议
   - 后续步骤指导
   - 风险提示

---

**注意**: 本Notebook提供市场环境判断功能，是陈小群战法的第一步。

**下一步**: 根据判断的情绪周期，选择合适的策略：
- **启动期** → 首板卡位术（见 `02_first_board_strategy.ipynb`）
- **加速期** → 龙头战法（见 `03_dragon_strategy.ipynb`）
- **过热期** → 逐步减仓（见 `04_reduce_position.ipynb`）
- **退潮期** → 空仓等待

---

**算法可靠性**: B级（中高可靠性）
- 基于陈小群游资战法知识库
- 多维度综合判断，置信度评分
- 数据可验证，逻辑清晰

## 💾 保存结果

本Notebook运行完成后，结果将自动保存到：
- **文件系统**: `notebooks/research/results/chen_xiaoqun_strategy/01_market_environment_judgment/YYYYMMDD_HHMMSS/`
- **MongoDB**: `jqquant.notebook_results` 集合

保存内容包括：
- ✅ 完整结果数据（JSON格式）
- ✅ 运行元数据（时间戳、参数等）
- ✅ 输出文本（如果有）
- ✅ 图表文件（如果有）
- ✅ Notebook副本（可选）

**后续引用**: 其他notebook可以通过运行ID或日期查询历史结果。

In [43]:
"""
保存notebook运行结果
自动保存到带时间戳的文件夹和MongoDB
"""

from core.notebook_result_manager import NotebookResultManager
from datetime import datetime, timezone, timedelta

print("=" * 80)
print("💾 保存Notebook运行结果")
print("=" * 80)

# 检查是否有result变量
if 'result' not in globals():
    print("⚠️  未找到 'result' 变量，请先运行前面的cell")
    print("   结果将不会被保存")
else:
    # 创建结果管理器
    manager = NotebookResultManager(
        strategy_name="chen_xiaoqun_strategy",
        notebook_name="01_market_environment_judgment"
    )
    
    # 准备保存的数据
    # 确保result包含所有关键信息
    save_result = result.copy() if isinstance(result, dict) else {'result': result}
    
    # 添加额外的元数据
    if 'limit_up_count' in globals():
        save_result['limit_up_count'] = limit_up_count
    if 'max_height' in globals():
        save_result['max_height'] = max_height
    if 'zhaban_rate' in globals():
        save_result['zhaban_rate'] = zhaban_rate
    if 'fund_attitude_score' in globals():
        save_result['fund_attitude_score'] = fund_attitude_score
    if 'sentiment_score' in globals():
        save_result['sentiment_score'] = sentiment_score
    
    # 准备参数
    parameters = {
        'notebook': '01_market_environment_judgment',
        'strategy': 'chen_xiaoqun_strategy',
        'run_time': datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    }
    
    # 保存结果
    try:
        save_info = manager.save_result(
            result=save_result,
            parameters=parameters,
            description="市场环境判断结果（情绪周期分析）",
            tags=["市场环境", "情绪周期", "陈小群战法"],
            save_notebook_copy=True
        )
        
        print(f"\n✅ 结果保存成功！")
        print(f"   运行ID: {save_info['run_id']}")
        print(f"   运行日期: {save_info['run_date']} {save_info['run_time']}")
        print(f"   文件路径: {save_info['file_path']}")
        print(f"   相对路径: {save_info['relative_path']}")
        if save_info.get('mongodb_id'):
            print(f"   MongoDB ID: {save_info['mongodb_id']}")
        print(f"   结果大小: {save_info['result_size'] / 1024:.2f} KB")
        print(f"   输出数量: {save_info['output_count']}")
        print(f"   图表数量: {save_info['chart_count']}")
        
        print(f"\n💡 后续引用方式:")
        print(f"   from core.notebook_result_manager import NotebookResultManager")
        print(f"   manager = NotebookResultManager('chen_xiaoqun_strategy', '01_market_environment_judgment')")
        print(f"   result = manager.load_result('{save_info['run_id']}')")
        
    except Exception as e:
        print(f"\n❌ 保存失败: {str(e)}")
        import traceback
        traceback.print_exc()

print("\n" + "=" * 80)

2026-01-14 09:25:37,553 - core.notebook_result_manager - INFO - MongoDB索引创建成功
2026-01-14 09:25:37,553 - core.notebook_result_manager - INFO - MongoDB连接成功: jqquant
2026-01-14 09:25:37,553 - core.notebook_result_manager - INFO - NotebookResultManager 初始化: chen_xiaoqun_strategy/01_market_environment_judgment
2026-01-14 09:25:37,553 - core.notebook_result_manager - INFO - 输出目录: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/results/chen_xiaoqun_strategy/01_market_environment_judgment
2026-01-14 09:25:37,554 - core.notebook_result_manager - INFO - 开始保存结果: 20260114_222537
2026-01-14 09:25:37,555 - core.notebook_result_manager - INFO - MongoDB保存成功: 6967a761a870f53f82a87ab6
2026-01-14 09:25:37,555 - core.notebook_result_manager - INFO - Notebook副本已保存: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/results/chen_xiaoqun_strategy/01_market_environment_judgment/20260114_222537/01_market_environment_judgment.ipynb
2026-01-14 09:25:37,555 - core.notebook_result_manager 

💾 保存Notebook运行结果

✅ 结果保存成功！
   运行ID: 20260114_222537
   运行日期: 2026-01-14 22:25:37
   文件路径: /home/taotao/.cursor/worktrees/TRQuant/ope/notebooks/research/results/chen_xiaoqun_strategy/01_market_environment_judgment/20260114_222537
   相对路径: chen_xiaoqun_strategy/01_market_environment_judgment/20260114_222537
   MongoDB ID: 6967a761a870f53f82a87ab6
   结果大小: 0.66 KB
   输出数量: 0
   图表数量: 0

💡 后续引用方式:
   from core.notebook_result_manager import NotebookResultManager
   manager = NotebookResultManager('chen_xiaoqun_strategy', '01_market_environment_judgment')
   result = manager.load_result('20260114_222537')

